# Scheduled Batch Inference Notebook

This notebook is intended to be scheduled in Amazon SageMaker Studio to perform batch inference on a model registered in MLflow. It downloads an input Parquet file from Amazon S3, loads the selected registered model version, and writes prediction results to an output Parquet file in Amazon S3.


## 1. Configure Parameters

When scheduling this notebook in SageMaker Studio, override these environment variables to customize input/output locations and the MLflow registered model selection. Defaults are provided for ad-hoc testing.


In [ ]:
import os

INPUT_DATA_S3_URI = os.environ.get("INPUT_DATA_S3_URI", "s3://my-input-bucket/path/to/input.parquet")
OUTPUT_DATA_S3_URI = os.environ.get("OUTPUT_DATA_S3_URI", "s3://my-output-bucket/path/to/predictions.parquet")
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "https://my-sagemaker-mlflow-domain")
MLFLOW_MODEL_NAME = os.environ.get("MLFLOW_MODEL_NAME", "my-registered-model")
MLFLOW_MODEL_STAGE = os.environ.get("MLFLOW_MODEL_STAGE", "Production")
MLFLOW_MODEL_VERSION = os.environ.get("MLFLOW_MODEL_VERSION")
LOCAL_WORKDIR = os.environ.get("LOCAL_WORKDIR", "/root/batch-inference")

print(f"Input data: {INPUT_DATA_S3_URI}")
print(f"Output data: {OUTPUT_DATA_S3_URI}")
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"MLflow model name: {MLFLOW_MODEL_NAME}")
print(f"MLflow stage: {MLFLOW_MODEL_STAGE}")
print(f"MLflow version: {MLFLOW_MODEL_VERSION}")
print(f"Local workdir: {LOCAL_WORKDIR}")


## 2. Install Dependencies (if needed)Most SageMaker notebook images include the required libraries. Uncomment and adjust the cell below if you need to install additional packages.

In [ ]:
# %%capture
# !pip install --quiet pyarrow pandas s3fs joblib


## 3. Imports and Utility Functions

The helper utilities below manage downloading and uploading objects in S3, schema enforcement, inference execution, and MLflow model resolution.


In [ ]:
import logging
from datetime import datetime
from pathlib import Path
from typing import Optional, Tuple
from urllib.parse import urlparse

import boto3
import mlflow
from mlflow import MlflowClient
from mlflow import pyfunc as mlflow_pyfunc
from mlflow import sklearn as mlflow_sklearn
from mlflow.exceptions import MlflowException
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

s3_resource = boto3.resource("s3")

INPUT_SCHEMA = {
    "offer_id": "Int64",
    "partner_id": "Int64",
    "product_id": "Int64",
    "carrier_code": "string",
    "flight_number": "Int64",
    "origination_code": "string",
    "destination_code": "string",
    "departure_timestamp": "datetime64[ns]",
    "seats_available": "Int64",
    "upgrade_type": "string",
    "item_count": "Int64",
    "usd_base_amount": "float64",
    "fare_class": "string",
    "from_cabin": "string",
    "created_timestamp": "datetime64[ns]",
    "multiplier_fare_class": "float64",
    "multiplier_loyalty": "float64",
    "multiplier_success_history": "float64",
    "multiplier_payment_type": "float64",
}

OUTPUT_COLUMNS = [
    "offer_id",
    "partner_id",
    "product_id",
    "carrier_code",
    "flight_number",
    "origination_code",
    "destination_code",
    "departure_timestamp",
    "upgrade_type",
    "accept_prob",
    "accept_prob_timestamp",
]


def parse_s3_uri(s3_uri: str) -> Tuple[str, str]:
    parsed = urlparse(s3_uri)
    if parsed.scheme != "s3" or not parsed.netloc or not parsed.path:
        raise ValueError(f"Invalid S3 URI: {s3_uri}")
    bucket = parsed.netloc
    key = parsed.path.lstrip("/")
    return bucket, key


def ensure_local_directory(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def download_s3_object(s3_uri: str, destination: Path) -> Path:
    bucket, key = parse_s3_uri(s3_uri)
    ensure_local_directory(destination.parent)
    logger.info("Downloading %s to %s", s3_uri, destination)
    s3_resource.Bucket(bucket).download_file(key, str(destination))
    return destination


def load_parquet(s3_uri: str, workdir: Path) -> pd.DataFrame:
    local_path = workdir / "input.parquet"
    download_s3_object(s3_uri, local_path)
    logger.info("Reading input parquet with schema enforcement")
    df = pd.read_parquet(local_path)
    for column, dtype in INPUT_SCHEMA.items():
        if column not in df.columns:
            raise KeyError(f"Expected column '{column}' missing from input data.")
        if dtype.startswith("datetime64"):
            df[column] = pd.to_datetime(df[column], errors="coerce")
        else:
            df[column] = df[column].astype(dtype)
    return df


def save_parquet(df: pd.DataFrame, s3_uri: str, workdir: Path) -> None:
    local_path = workdir / "predictions.parquet"
    ensure_local_directory(local_path.parent)
    df.to_parquet(local_path, index=False)
    bucket, key = parse_s3_uri(s3_uri)
    logger.info("Uploading predictions to %s", s3_uri)
    s3_resource.Bucket(bucket).upload_file(str(local_path), key)


def resolve_model_target(
    client: MlflowClient,
    model_name: str,
    stage: Optional[str],
    version: Optional[str],
) -> Tuple[str, Optional[str], str]:
    if version:
        model_version = client.get_model_version(name=model_name, version=str(version))
        model_uri = f"models:/{model_name}/{model_version.version}"
        return model_version.version, model_version.current_stage or None, model_uri

    target_stage = (stage or "Production").strip()
    if not target_stage:
        raise MlflowException(
            "MLFLOW_MODEL_STAGE must be provided when MLFLOW_MODEL_VERSION is not set."
        )
    latest_versions = client.get_latest_versions(model_name, stages=[target_stage])
    if not latest_versions:
        raise MlflowException(
            f"No versions found for model '{model_name}' in stage '{target_stage}'."
        )
    model_version = latest_versions[0]
    model_uri = f"models:/{model_name}/{target_stage}"
    return model_version.version, model_version.current_stage or target_stage, model_uri


def load_registered_model(
    client: MlflowClient,
    model_name: str,
    stage: Optional[str],
    version: Optional[str],
):
    resolved_version, resolved_stage, model_uri = resolve_model_target(
        client=client,
        model_name=model_name,
        stage=stage,
        version=version,
    )
    try:
        model = mlflow_sklearn.load_model(model_uri)
    except Exception:
        logger.warning("Falling back to MLflow pyfunc loader for %s", model_uri)
        model = mlflow_pyfunc.load_model(model_uri)
    return model, resolved_version, resolved_stage, model_uri


def compute_accept_probabilities(model, features: pd.DataFrame) -> pd.Series:
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(features)[:, 1]
        return pd.Series(probabilities, index=features.index, dtype="float64")

    predictions = model.predict(features)
    if isinstance(predictions, pd.DataFrame):
        if "accept_prob" in predictions.columns:
            return predictions["accept_prob"].astype("float64")
        first_col = predictions.columns[0]
        return predictions[first_col].astype("float64")

    return pd.Series(predictions, index=features.index, dtype="float64")


def run_inference(model, enriched_input: pd.DataFrame) -> pd.DataFrame:
    logger.info("Running model inference on %d rows", len(enriched_input))
    feature_columns = [
        col
        for col in enriched_input.columns
        if col
        not in {
            "offer_id",
            "partner_id",
            "product_id",
            "carrier_code",
            "flight_number",
            "origination_code",
            "destination_code",
            "departure_timestamp",
            "upgrade_type",
        }
    ]
    probabilities = compute_accept_probabilities(model, enriched_input[feature_columns])
    output = enriched_input[
        [
            "offer_id",
            "partner_id",
            "product_id",
            "carrier_code",
            "flight_number",
            "origination_code",
            "destination_code",
            "departure_timestamp",
            "upgrade_type",
        ]
    ].copy()
    output["accept_prob"] = probabilities
    output["accept_prob_timestamp"] = datetime.utcnow()
    return output[OUTPUT_COLUMNS]


## 4. Execute Batch Inference

The following cell orchestrates the download, inference, and upload steps. It resolves the requested model version from the MLflow registry, loads it into memory, generates predictions, and persists them back to S3. Rerun this cell (or schedule the notebook) whenever predictions need to be refreshed.


In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow_client = MlflowClient()

workdir = Path(LOCAL_WORKDIR)
ensure_local_directory(workdir)

input_df = load_parquet(INPUT_DATA_S3_URI, workdir)
model, resolved_version, resolved_stage, model_uri = load_registered_model(
    client=mlflow_client,
    model_name=MLFLOW_MODEL_NAME,
    stage=None if MLFLOW_MODEL_VERSION else MLFLOW_MODEL_STAGE,
    version=MLFLOW_MODEL_VERSION,
)
logger.info(
    "Loaded MLflow model '%s' version %s (stage: %s) from %s",
    MLFLOW_MODEL_NAME,
    resolved_version,
    resolved_stage or "n/a",
    model_uri,
)

predictions_df = run_inference(model, input_df)
save_parquet(predictions_df, OUTPUT_DATA_S3_URI, workdir)
logger.info("Batch inference complete. %d predictions written.", len(predictions_df))


## 5. (Optional) Cleanup Local Artifacts

If your job container has limited storage, uncomment the cleanup cell below to remove temporary files after the run.


In [ ]:
# import shutil
# shutil.rmtree(LOCAL_WORKDIR, ignore_errors=True)


## 6. Schedule This Notebook in SageMaker Studio

1. In SageMaker Studio, open this notebook and choose **Run > Schedule Notebook**.
2. Provide a job name, execution role, and select the image/kernel that contains the required dependencies (for example, a Python 3 Data Science image).
3. Set the **Start time** and **Repeat** cadence that matches how often you need refreshed predictions.
4. Define the environment variables listed in the parameter cell (for example `INPUT_DATA_S3_URI`, `OUTPUT_DATA_S3_URI`, `MLFLOW_TRACKING_URI`, `MLFLOW_MODEL_NAME`, `MLFLOW_MODEL_STAGE` or `MLFLOW_MODEL_VERSION`, and `LOCAL_WORKDIR`).
5. Optionally enable output logging to Amazon CloudWatch Logs for easier troubleshooting.
6. Save the schedule. SageMaker Studio will launch the notebook on the defined cadence and persist the generated Parquet predictions to the configured S3 location.
